# 00 — Threads: OS-Level Concurrency

**Goal:** Understand OS threads, the GIL, and why asyncio was invented.

## Threads vs Coroutines

| | Threads (OS) | Coroutines (asyncio) |
|---|---|---|
| Who switches? | **OS** (preemptive) | **You** at `await` (cooperative) |
| Can be interrupted mid-line? | **YES** (dangerous!) | **NO** (safe between awaits) |
| Memory overhead | ~8MB stack per thread | ~1KB per coroutine |
| Max concurrent | ~hundreds | ~tens of thousands |
| GIL? | Yes — only 1 thread runs Python at a time | N/A — single thread |

## Exercise 0.1 — Your first threads

Write a `worker(name, seconds)` function that prints a start message (with thread name from `threading.current_thread().name`), sleeps for `seconds`, then prints done.

Create 3 threads with different sleep durations (2, 1, 3 seconds). Start all, then `.join()` all.

Time the whole thing — should be ~3s (max), not 6s (sum).

**Why does this work despite the GIL?** `time.sleep()` releases the GIL. During I/O, the thread gives up the GIL so other threads can run.

In [ ]:
import threading
import time

# Exercise 0.1 — your code here


### Question 0.1
Total time is ~3s, not 6s. But Python has the GIL — how can threads run concurrently?

*Your answer:*


## Exercise 0.2 — The GIL in action: CPU-bound failure

Write a `cpu_work(n)` function that does pure computation (e.g., sum `i*i` for `i` in `range(n)`). No I/O.

With `N = 10_000_000`:
1. Time two sequential calls: `cpu_work(N); cpu_work(N)`
2. Time two threaded calls: start both threads, join both
3. Compute the speedup ratio

Expected: speedup ~1x (or even <1x due to GIL contention). Threading gives NO speedup for CPU-bound work.

**Decision rule:**
```
I/O-bound:  → asyncio (best) or threading (ok)
CPU-bound:  → multiprocessing (Module 03)
```

In [ ]:
# Exercise 0.2 — your code here


## Exercise 0.3 — Daemon threads and thread lifecycle

Write a `background_job()` that loops 10 times with `time.sleep(0.3)` between each.

Run it as:
1. A normal thread — observe that `t.join()` is needed to wait for it
2. A daemon thread (`daemon=True`) — in a real script, killed when main thread exits

Understand: daemon threads are for fire-and-forget background work.

In [ ]:
# Exercise 0.3 — your code here


## Summary

| Concept | Details |
|---------|--------|
| `threading.Thread` | Creates an OS thread |
| `.start()` / `.join()` | Begin / wait for thread |
| `daemon=True` | Thread dies when main exits |
| **GIL** | Only 1 thread runs Python bytecode at a time |
| **GIL release** | During I/O, sleep, C extensions |
| **CPU-bound** | Threads DON'T help → use multiprocessing |

**Next notebook:** Shared state bugs — race conditions and why threads are dangerous